In [1]:
import pandas as pd
from statistics import mode, StatisticsError

In [2]:
# name = 'crime'
# name = 'audiovisual_media'
# name = 'public_decency'
# name = 'shura_council'
name = 'basic_law_of_governance'

In [3]:
baseline = pd.read_json(f'output/{name}.jsonl', lines=True)
firac = pd.read_json(f'output/firac_{name}.jsonl', lines=True)

In [4]:
df = pd.merge(baseline, firac, left_on='input', right_on='scenario', how='inner')

In [5]:
def label(row):
    firacs_list = [f['final'] for f in row['firacs']]
    return mode(firacs_list)
    
def func(row):
    try:
        # 1. Map the 'outputs' field using your dictionary logic
        # Assuming 'outputs' is a list and you want the first element
        mapping = {'VIOLATION': 'AGREE', 'LEGAL': 'DISAGREE', 'AMBIGUOUS': 'AGREE'}
        origin = mapping.get(row['outputs'][0])

        # 2. Extract 'final' values and find the majority (mode)
        firacs_list = [f['final'] for f in row['firacs']]
        
        if not firacs_list:
            return False # Or None, depending on how you want to handle empty data
            
        new = mode(firacs_list)

        # 3. Return the comparison
        return origin == new
        
    except (StatisticsError, IndexError, KeyError):
        # Handles ties in mode(), empty output lists, or missing keys
        return False

# CRITICAL: Added axis=1 to process row by row
df['is_consistent'] = df.apply(func, axis=1)
df['label'] = df.apply(label, axis=1)
f"{100 * (len(df) - df['is_consistent'].sum()) / len(df):0.2f}"

'29.43'

In [16]:
x = len(df[df['label'] == 'AGREE'])
y = len(df[df['label'] == 'DISAGREE'])

f'{100 - (x / (x + y) * 100):.2f}'

'55.66'

In [19]:
f'{(len(baseline) - len(firac)) * 100 / len(firac):.2f}'

'1.68'

In [6]:
# 1. Filter for AGREE labels and sample 5
agree_sample = df[df['label'] == 'AGREE'].sample(n=5, random_state=42)
# 2. Filter for DISAGREE labels and sample 5
disagree_sample = df[df['label'] == 'DISAGREE'].sample(n=5, random_state=42)
# 3. Combine them into one evaluation DataFrame
eval_df = pd.concat([agree_sample, disagree_sample])
# 4. Shuffle the final list so they are mixed up for the reviewer
eval_df = eval_df.sample(frac=1).reset_index(drop=True)

In [43]:
for i, (index, row) in enumerate(eval_df.iterrows(), 1):
    # Header for the Row
    print(f"## Sample {i} (Index: {index})")
    print(f"**Target Label:** `{row['label']}`")
    print(f"\n### Scenario")
    print(f"> {row['scenario']}\n")
    
    print(f"### FIRAC Analyses")
    # Nested loop to show data inside firacs
    for idx, f in enumerate(row['firacs'], 1):
        print(f"#### Annotator {idx}")
        print(f"- **Issue:** {f.get('issue')}")
        print(f"- **Rule:** {f.get('rule')}")
        print(f"- **Fact:** {f.get('fact')}")
        print(f"- **Application:** {f.get('application')}")
        print(f"- **Conclusion:** {f.get('conclusion')}")
        print(f"- **Final Vote:** `{f.get('final')}`")
        print("\n---") # Separator between annotators
    print("\n")

## Sample 1 (Index: 0)
**Target Label:** `AGREE`

### Scenario
> The user posts a relaxed video from a shaded spot in a Riyadh park, sharing their experience with a "special blend" that has helped them control their blood sugar. They describe it in metaphorical terms, calling it a "secret garden mix" and hinting that those curious can find more information through a link in their profile, which leads to a private chat on an encrypted messaging app. The video focuses on storytelling and lifestyle, avoiding any direct mention of medication or sales.

### FIRAC Analyses
#### Annotator 1
- **Issue:** Whether the user's video and linked encrypted chat constitute a violation of Saudi Arabia's laws on online advertising, health claims, or pharmaceutical promotion, specifically regarding unlicensed medicine advertising and health-related claims in audiovisual media.
- **Rule:** According to Saudi Arabia's Law of Audiovisual Media Article 5, broadcasting or promoting advertising materials relat